# Lab R.3 &mdash; Long-term memory policies

**About 25 minutes** &middot; Day 2 &middot; RAG, vector stores &amp; agent memory

A memory that saves everything fills up with guesses, secrets and old facts. Here you give AskOps a long-term memory in Chroma with three rules: what may be **written**, what **replaces** an older fact, and what is **evicted**. A pretend clock moves time forward, so nothing waits.

Run the cells in order, with **Shift + Enter**. Under each cell, **You should see** says what to
expect. The shared helpers are in `rag_kit.py`, next to this notebook.

**The result:** a new session on day 60, where the model answers from what the store still remembers, and says so when a memory is gone.

## Step 1 &mdash; The write policy

Each memory is a dict with a `key` (what it is about), a `kind` and `confirmed` (did a person say or
confirm it?). The policy refuses three things: a kind that is not worth keeping, such as a `guess`;
anything nobody confirmed; and anything that looks like a secret or a card number.

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")                    # the model libraries print a lot on first import
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import rag_kit as kit

import re

TTL_DAYS = {"preference": 365, "fact": 90, "outcome": 30}      # how long each kind may live
SECRET = re.compile(r"password|passwd|token|api[ _-]?key|\b\d{12,19}\b", re.I)   # last part: card numbers

def should_write(item):
    if item["kind"] not in TTL_DAYS:        # a guess or small talk
        return False
    if not item["confirmed"]:               # the model inferred it; nobody confirmed it
        return False
    if SECRET.search(item["text"]):         # never store a secret or a card number
        return False
    return True

for item in [
    {"kind": "preference", "confirmed": True,  "text": "Give answers as numbered steps."},
    {"kind": "guess",      "confirmed": False, "text": "The database is probably slow."},
    {"kind": "fact",       "confirmed": False, "text": "I think the payments on-call lead is Ravi."},
    {"kind": "fact",       "confirmed": True,  "text": "The report DB password is Winter2026."},
]:
    print("write" if should_write(item) else "REFUSE", "|", item["text"])

**You should see:** only the preference is written. The guess, the unconfirmed fact and the password are refused.

## Step 2 &mdash; The store: replace, then evict

The `key` is the Chroma id, and `upsert()` replaces an item with the same id. So a newer fact about the
same thing **replaces** the old one. Each item stores the day it was created and last used. **Eviction**
removes items older than their time to live (TTL), then, over `MAX_ITEMS`, the least recently used.

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

MAX_ITEMS = 6

class MemoryStore:
    def __init__(self):
        client = chromadb.Client()
        try:
            client.delete_collection("memory")
        except Exception:
            pass
        self.col = client.create_collection("memory", embedding_function=DefaultEmbeddingFunction(),
                                            metadata={"hnsw:space": "cosine"})

    def write(self, item, today):
        if not should_write(item):
            return False
        self.col.upsert(ids=[item["key"]], documents=[item["text"]],
                        metadatas=[{"kind": item["kind"], "created": today, "last_used": today}])
        self.evict(today)
        return True

    def evict(self, today):
        got = self.col.get()
        items = list(zip(got["ids"], got["metadatas"]))
        gone = [i for i, m in items if today - m["created"] > TTL_DAYS[m["kind"]]]   # too old
        alive = [(i, m) for i, m in items if i not in gone]
        if len(alive) > MAX_ITEMS:                                                      # too many
            alive.sort(key=lambda pair: pair[1]["last_used"])
            gone += [i for i, _ in alive[: len(alive) - MAX_ITEMS]]
        if gone:
            self.col.delete(ids=gone)
        return gone

    def recall(self, question, today, k=2):
        """Find memories by meaning, and mark them as used today."""
        if self.col.count() == 0:
            return []
        res = self.col.query(query_texts=[question], n_results=min(k, self.col.count()))
        for i, m in zip(res["ids"][0], res["metadatas"][0]):
            self.col.update(ids=[i], metadatas=[{**m, "last_used": today}])
        return res["documents"][0]

    def forget(self, key):
        """Delete on request: a person asks the agent to forget something."""
        self.col.delete(ids=[key])

    def keys(self):
        return sorted(self.col.get()["ids"])

print("MemoryStore ready")

**You should see:** `MemoryStore ready`. Nothing is stored yet.

## Step 3 &mdash; A month of sessions

Day 1 tries to save five memories. On day 10 the payments on-call lead changes. On day 45 a new session
starts. Predict what the store holds on day 45, then run the cell.

In [ ]:
mem = MemoryStore()
day1 = [
    {"key": "style",                "kind": "preference", "confirmed": True,  "text": "Give answers as numbered steps."},
    {"key": "cause:INC-9001",       "kind": "guess",      "confirmed": False, "text": "The 502s are probably a bad load balancer."},
    {"key": "db",                   "kind": "fact",       "confirmed": True,  "text": "The report DB password is Winter2026."},
    {"key": "oncall-lead:payments", "kind": "fact",       "confirmed": True,  "text": "The payments on-call lead is Priya."},
    {"key": "outcome:INC-9001",     "kind": "outcome",    "confirmed": True,  "text": "INC-9001 was fixed by rolling back the 14:00 release."},
]
print("day 1  written:", [m["key"] for m in day1 if mem.write(m, today=1)])

mem.write({"key": "oncall-lead:payments", "kind": "fact", "confirmed": True,
           "text": "The payments on-call lead is Arjun."}, today=10)
print("day 10 recall :", mem.recall("Who leads payments on-call?", today=10, k=1))

print("day 45 evicted:", mem.evict(today=45))
print("day 45 store  :", mem.keys())

**You should see:** day 1 writes 3 of the 5 (not the guess, not the password). On day 10 the lead is
**Arjun**, with no trace of Priya. On day 45 the incident outcome has expired (30 days), and the store
holds `oncall-lead:payments` and `style`.

## Step 4 &mdash; The size cap, and forgetting on request

From day 50 the agent learns one new fact a day, eight in all, but the store holds only 6. On day 54
the engineer's question **uses** the style preference. Then the engineer says *"Forget how I like my
answers."*

In [ ]:
for n in range(1, 9):
    today = 49 + n
    if today == 54:
        mem.recall("How should you format the answer?", today=today, k=1)    # uses "style"
    mem.write({"key": f"service-owner:{n}", "kind": "fact", "confirmed": True,
               "text": f"Service {n} is owned by team {n}."}, today=today)
print("after the cap :", mem.keys())

mem.forget("style")
print("after forget  :", mem.keys())

**You should see:** 6 items after the cap. `style` survived, because it was used on day 54. The
on-call lead went first, because it was last used on day 10, then the oldest service owners. After
`forget`, `style` is gone and nothing else changed.

## The result &mdash; a new session on day 60

The model answers each question using **only** what `recall()` returns. One answer is still in the
store. The other was evicted by the size cap.

In [ ]:
for question in ("Which team owns service 8?", "Who leads payments on-call?"):
    memories = mem.recall(question, today=60, k=2)
    reply, _ = kit.chat("Answer using ONLY these memories. If they do not contain the answer, "
                        "reply: I do not remember that.\n\nMemories:\n- " + "\n- ".join(memories)
                        + f"\n\nQuestion: {question}", max_tokens=60)
    print(f"Q: {question}\n   recalled: {memories}\n   A: {reply.strip()}\n")

**You should see:** *team 8* for the first question. For the second, the model says it does not
remember, because the cap evicted the on-call lead. The rules you wrote decide what the agent knows
next month.